# HW3: PPO в среде с непрерывными действиями

Среда: **HalfCheetah-v5** (Gymnasium + MuJoCo). Алгоритм: **PPO**. Три прогона с разными гиперпараметрами; график средней награды; выводы.


In [ ]:
!pip install -q "gymnasium[mujoco]"

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.distributions.normal import Normal
import random
import time

import matplotlib.pyplot as plt
from IPython.display import clear_output

def make_env(env_id, idx):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        return env
    return thunk

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs, continue_action=False):
        super().__init__()
        self.continue_action = continue_action
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        if self.continue_action:
            self.actor_mean = nn.Sequential(
                layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
                nn.Tanh(),
                layer_init(nn.Linear(64, 64)),
                nn.Tanh(),
                layer_init(nn.Linear(64, np.prod(envs.single_action_space.shape)), std=0.01),
            )
            self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)))
        else:
            self.actor = nn.Sequential(
                layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
                nn.Tanh(),
                layer_init(nn.Linear(64, 64)),
                nn.Tanh(),
                layer_init(nn.Linear(64, envs.single_action_space.n), std=0.01),
            )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None, deterministic=False):
        if self.continue_action:
            action_mean = self.actor_mean(x)
            action_logstd = self.actor_logstd.expand_as(action_mean)
            action_std = torch.exp(action_logstd)
            probs = Normal(action_mean, action_std)
            if action is None:
                action = probs.sample()
            return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(x)
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

class Args:
    seed: int = 1
    torch_deterministic: bool = True
    cuda: bool = True
    env_id: str = "HalfCheetah-v5"
    total_timesteps: int = 400_000
    learning_rate: float = 2.5e-4
    num_envs: int = 8
    num_steps: int = 128
    anneal_lr: bool = True
    gamma: float = 0.99
    gae_lambda: float = 0.95
    num_minibatches: int = 4
    update_epochs: int = 4
    norm_adv: bool = True
    clip_coef: float = 0.2
    clip_vloss: bool = True
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0


## Гиперпараметры экспериментов

1. **Базовый** — типичные значения (как в CleanRL / SB3): `learning_rate=2.5e-4`, `clip_coef=0.2`, `ent_coef=0.01`.
2. **Большой шаг** — `learning_rate=1e-3`: сильнее меняются веса, возможны более быстрые ранние улучшения или нестабильность.
3. **Узкий клип, без бонуса энтропии** — `clip_coef=0.1`, `ent_coef=0`: более осторожные обновления политики и меньше стимула к исследованию.


In [ ]:
def show_progress(log_train_reward, log_entropy, log_loss):
    clear_output(True)
    plt.figure(figsize=[12, 4])
    ax1 = plt.subplot(1, 3, 1)
    ax1.plot(list(zip(*log_train_reward))[0], color="red")
    ax1.set_title("mean train return")
    ax1.grid()
    ax2 = plt.subplot(1, 3, 2)
    ax2.plot(list(zip(*log_entropy))[0], color="blue")
    ax2.set_title("entropy")
    ax2.grid()
    ax3 = plt.subplot(1, 3, 3)
    ax3.plot(list(zip(*log_loss))[0], color="green")
    ax3.set_title("loss")
    ax3.grid()
    plt.tight_layout()
    plt.show()

def train_ppo_run(run_name, lr, clip_c, ent_c):
    args = Args()
    args.learning_rate = lr
    args.clip_coef = clip_c
    args.ent_coef = ent_c
    args.batch_size = int(args.num_envs * args.num_steps)
    args.minibatch_size = int(args.batch_size // args.num_minibatches)
    args.num_iterations = args.total_timesteps // args.batch_size
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic
    device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")
    envs = gym.vector.SyncVectorEnv([make_env(args.env_id, i) for i in range(args.num_envs)])
    continue_action = isinstance(envs.single_action_space, gym.spaces.Box)
    agent = Agent(envs, continue_action=continue_action).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)
    obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
    logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
    rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
    dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
    values = torch.zeros((args.num_steps, args.num_envs)).to(device)
    next_obs, _ = envs.reset(seed=args.seed)
    next_obs = torch.Tensor(next_obs).to(device)
    next_done = torch.zeros(args.num_envs).to(device)
    train_rewards = []
    log_train_reward = []
    losses = []
    log_loss = []
    entropies = []
    log_entropy = []
    for iteration in range(1, args.num_iterations + 1):
        if args.anneal_lr:
            frac = 1.0 - (iteration - 1.0) / args.num_iterations
            optimizer.param_groups[0]["lr"] = frac * args.learning_rate
        for step in range(args.num_steps):
            obs[step] = next_obs
            dones[step] = next_done
            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()
            actions[step] = action
            logprobs[step] = logprob
            act_np = np.clip(action.cpu().numpy(), envs.single_action_space.low, envs.single_action_space.high)
            next_obs, reward, terminations, truncations, infos = envs.step(act_np)
            np_next_done = np.logical_or(terminations, truncations)
            rewards[step] = torch.tensor(reward).to(device).view(-1)
            next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(np_next_done).to(device)
            if "episode" in infos:
                train_rewards.append((infos["episode"]["r"] * np_next_done).sum() / np_next_done.sum())
        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0
            for t in reversed(range(args.num_steps)):
                if t == args.num_steps - 1:
                    nextnonterminal = 1.0 - next_done
                    nextvalues = next_value
                else:
                    nextnonterminal = 1.0 - dones[t + 1]
                    nextvalues = values[t + 1]
                delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
                advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
            returns = advantages + values
        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)
        b_inds = np.arange(args.batch_size)
        for epoch in range(args.update_epochs):
            np.random.shuffle(b_inds)
            for start in range(0, args.batch_size, args.minibatch_size):
                end = start + args.minibatch_size
                mb_inds = b_inds[start:end]
                mb_act = b_actions[mb_inds]
                if continue_action:
                    _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], mb_act)
                else:
                    _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], mb_act.long())
                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()
                mb_advantages = b_advantages[mb_inds]
                if args.norm_adv:
                    mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()
                newvalue = newvalue.view(-1)
                if args.clip_vloss:
                    v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                    v_clipped = b_values[mb_inds] + torch.clamp(
                        newvalue - b_values[mb_inds], -args.clip_coef, args.clip_coef)
                    v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                    v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped).mean()
                else:
                    v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()
                entropy_loss = entropy.mean()
                loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
                optimizer.step()
                losses.append(loss.item())
                entropies.append(entropy_loss.item())
        mean_train_reward = float(np.mean(train_rewards)) if train_rewards else 0.0
        mean_loss = float(np.mean(losses)) if losses else 0.0
        mean_entropy = float(np.mean(entropies)) if entropies else 0.0
        log_train_reward.append([mean_train_reward])
        log_entropy.append([mean_entropy])
        log_loss.append([mean_loss])
        show_progress(log_train_reward, log_entropy, log_loss)
    envs.close()
    return run_name, list(zip(*log_train_reward))[0]

all_runs = {}
for name, lr, clip_c, ent_c in [
    ("default", 2.5e-4, 0.2, 0.01),
    ("lr_1e-3", 1e-3, 0.2, 0.01),
    ("clip0.1_ent0", 2.5e-4, 0.1, 0.0),
]:
    k, curve = train_ppo_run(name, lr, clip_c, ent_c)
    all_runs[k] = curve


In [ ]:
plt.figure(figsize=(9, 4))
for k, v in all_runs.items():
    plt.plot(v, label=k)
plt.xlabel("iteration")
plt.ylabel("mean of finished episode returns (see code)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


## Выводы

Сопоставьте три кривые: если при большом `learning_rate` награда растёт быстрее в начале, но потом колеблется — это согласуется с более агрессивным градиентным шагом. При `clip_coef=0.1` и `ent_coef=0` обновления политики сдержаннее и исследование слабее: кривая может быть гладче или, наоборот, медленнее выходить на высокий уровень. Конкретная картина зависит от числа шагов, сида и железа; опишите фактическое поведение ваших графиков в 2–4 предложениях перед сдачей.
